# Kisan village matcher

**The caller says "Nagar". The roster says "Ahilyanagar". Nobody may guess.**

A kisan call centre agent takes a call from a farmer whose crop insurance claim is stuck. The
farmer says where they are, in Marathi or Telugu or Bhojpuri-accented Hindi, shortened the way
people actually shorten names. On the agent's screen is a roster of official names in the native
script. Between the two sits every way this can go wrong: a different script, a different
romanization, a name that changed in 2024, or a name that two states both use.

A wrong match is a claim filed against the wrong district office. So the rule is not "match well".
It is **under confidence, ask**.

## Read this first

**This notebook has not been run against the live Sarvam API.** There was no API key on the
machine where it was written, so every cell below ships with an empty output. The two cells that
call Sarvam are written and unrun. Run them yourself with your own key before trusting them.

**The roster is a demonstrative sample of 48 district entries across 11 states**, authored by hand
from published government notifications and press reports. It is not a gazetteer of India.

## The pipeline

    audio clip
        |
        +-- speech_to_text.transcribe(mode="transcribe")  ->  native script, for the agent to read
        +-- speech_to_text.transcribe(mode="translit")    ->  Latin script, for the matcher to score
                                                                 |
                                          fold  ->  score  ->  rank  ->  band
                                                                 |
                                   MATCH: one district   ASK: a question   NO_MATCH: nothing

In [ ]:
%pip install -q "sarvamai>=0.1.24" "python-dotenv>=1.0.0"

## Setup

The matching half of this recipe needs no key and no network. Only the two Sarvam cells near the
bottom do, and they read the key at call time rather than at import time. `SarvamAI.__init__` takes
its key as a default argument, which Python evaluates once when the package is imported, so a key
exported after that import is never seen. Passing it explicitly is the only reliable way.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

RECIPE_DIR = Path.cwd()
if str(RECIPE_DIR) not in sys.path:
    sys.path.insert(0, str(RECIPE_DIR))

load_dotenv(RECIPE_DIR / ".env")

import village_matcher as vm


def require_key() -> str:
    """Fail loudly and early rather than deep inside an SDK call."""
    key = os.environ.get("SARVAM_API_KEY", "")
    if not key:
        raise RuntimeError(
            "SARVAM_API_KEY is not set. Copy .env.example to .env, put your key in it, "
            "and re-run this cell. The offline cells below run without it."
        )
    return key


print(vm.ROSTER_SIZE, "roster entries,",
      len({p.state for p in vm.ROSTER}), "states,",
      vm.RENAMES_SIZE, "renames")

## 1. Folding: making two spellings of one place into one string

`fold()` reduces a Latin place name to a comparison key. Lowercase, punctuation and separators
gone, administrative words like "district" and "taluka" dropped, then 13 spelling rules applied in
a fixed order.

The words `nagar`, `dehat`, `urban` and `rural` are deliberately **not** dropped. Four real
districts are told apart by nothing else.

In [ ]:
variants = [
    ("Ahilyanagar", "Ahilya Nagar", "AHILYANAGAR", "Ahilya-Nagar", "Ahilyanagar District"),
    ("Nashik", "Nasik"),
    ("Vijayapura", "Vijayapur"),
    ("Kolhapur", "Kolhapore"),
    ("Warangal", "Varangal"),
    ("Bengaluru", "Bengalooru"),
    ("Beed", "Bid"),
    ("Lucknow", "Luknow"),
]

for group in variants:
    folded = {vm.fold(s) for s in group}
    print(f"{str(group):<80} -> {sorted(folded)}")

### The rules that are absent matter more

There is no rule collapsing double letters and no rule collapsing aspiration. Either one would
merge two different real places in two different states. This is the half of the folding layer
that carries the risk.

In [ ]:
for a, b, why, provenance in vm.MINIMAL_PAIRS:
    same = "MERGED" if vm.fold(a) == vm.fold(b) else "kept apart"
    print(f"{a:<18} {vm.fold(a):<16} {b:<18} {vm.fold(b):<16} {same:<11} {why} ({provenance})")

## 2. The three answers

`match()` ranks every roster entry, then classifies the result into one of three bands.

| Band | Meaning | What the agent does |
|---|---|---|
| `MATCH` | one place, decisively | proceed |
| `ASK` | more than one plausible reading | read the question back to the caller |
| `NO_MATCH` | nothing worth offering | ask the caller to repeat, offer nothing |

The queries below are **authored example spellings, not recorded transcripts**. Nobody said these
into a microphone. They are written the way an agent or a speech recogniser plausibly would write
them, and they exist to drive the offline core. The real input is the `translit` transcript from
the audio cell further down.

In [ ]:
SPOKEN_FIXTURES = (
    "Ahilyanagar",
    "Ahmednagar",
    "Tumkur",
    "Nasik",
    "Bid",
    "Kolhapore",
    "Simla",
    "Warangal",
    "Vijayapur",
    "Nagar",
    "Bilaspur",
    "Aurangabad",
    "Bengaluru",
    "Kanpur",
    "Zzzqqqxx",
)

for spoken in SPOKEN_FIXTURES:
    result = vm.match(spoken)
    top = result.candidates[0] if result.candidates else None
    summary = f"{top.place.name} ({top.place.state}) {top.score:.3f}" if top else "-"
    print(f"{result.band:<9} {spoken:<14} {summary}")
    if result.question:
        print(f"{'':>9} {'':<14} {result.question}")

## 3. Why a high score is not enough

`"Nagar"` scores **0.909** against Nagaur in Rajasthan, clear of the 0.90 match threshold, with the
runner-up 0.18 behind. A threshold-only matcher would have returned a confident wrong district in a
state the caller has never been to, and said nothing about it.

Two guards stop that. The longest block the two names share has to be at least `MIN_ANCHOR`
characters, and it has to cover at least `MIN_CANDIDATE_COVERAGE` of the name that matched.
"Nagar" shares 4 characters with "nagaur", which is 4 of 6, or 0.667. Below the floor, so the band
drops to ASK and the caller is asked.

In [ ]:
result = vm.match("Nagar")
top = result.candidates[0]
anchor = vm.anchor_size(result.folded, top.matched)

print("query folded to :", result.folded)
print("top candidate   :", top.place.name, f"({top.place.state})", f"{top.score:.3f}")
print("above threshold :", top.score >= vm.MATCH_THRESHOLD)
print("shared block    :", anchor, "characters, floor is", vm.MIN_ANCHOR)
print("coverage        :", f"{anchor / len(top.matched):.3f}", "floor is", vm.MIN_CANDIDATE_COVERAGE)
print("band            :", result.band)
print("question        :", result.question)

## 4. Names that changed, and why a rename is not a find-and-replace

Ahmednagar became Ahilyanagar in 2024. A farmer who has said "Ahmednagar" their whole life will
keep saying it. Without the renames table, fuzzy matching alone ranks **Karimnagar in Telangana**
above Ahilyanagar in Maharashtra for that query, which is a farmer sent 500 km to the wrong state.

Each rename is keyed to the government that notified it. Maharashtra renamed its Aurangabad to
Chhatrapati Sambhajinagar in 2023; Bihar's Aurangabad district was not renamed. Ask about
"Aurangabad" and both readings come back, and the recipe asks which one you mean.

In [ ]:
for query in ("Ahmednagar", "Tumkur"):
    for label, renames in (("with renames", vm.RENAMES), ("without renames", ())):
        result = vm.match(query, renames=renames)
        top = result.candidates[0]
        print(f"{query:<12} {label:<16} {result.band:<9} "
              f"{top.place.name} ({top.place.state}) {top.score:.3f}  via {top.via}")
    print()

aurangabad = vm.match("Aurangabad")
print(aurangabad.band)
for candidate in aurangabad.candidates[:2]:
    print(f"  {candidate.place.name} ({candidate.place.state}) "
          f"{candidate.score:.3f}  via {candidate.via}")
print(aurangabad.question)

## 5. One clip, two projections

Everything above ran with no key. This is where Sarvam comes in, and it does exactly two things.

The first is turning one audio clip into two readings of the same utterance, from the same
endpoint, differing only in `mode`:

- `mode="transcribe"` returns the utterance in its own script. That is what the agent should see.
- `mode="translit"` returns romanization of the same audio. That is what the matcher folds and
  scores.

`mode` is documented as applying to `saaras:v3`, which is also the model this repo's allowlist
permits, so `saaras:v3` is what `sarvam_projection.py` sends.

**No audio ships with this recipe.** Record a short clip of somebody saying a district name, put it
in `sample_data/` (gitignored, so it stays on your machine) and point `CLIP` at it. The cell below
has not been run.

In [ ]:
import sarvam_projection

CLIP = RECIPE_DIR / "sample_data" / "village_name.wav"

if not CLIP.exists():
    print(f"No clip at {CLIP}. Drop a recording there to run this cell.")
else:
    require_key()
    native_text, latin_text = sarvam_projection.transcribe_both_ways(CLIP, language_code="mr-IN")
    print("native script :", native_text)
    print("romanized     :", latin_text)

    result = vm.match(latin_text)
    print("band          :", result.band)
    for candidate in result.candidates:
        print(f"  {candidate.place.name} ({candidate.place.state}) {candidate.score:.3f}")
    if result.question:
        print(result.question)

## 6. Projecting the roster into Latin

The second Sarvam call transliterates a native-script roster name to `en-IN`. Transliteration
accepts 11 language codes where speech recognition accepts 24, so every roster entry already
carries its own Latin name in the data and the offline core never needs this call. It is useful for
checking a roster you brought yourself, in a language transliteration supports.

This cell has not been run either.

In [ ]:
sample = vm.ROSTER[:5]

if not os.environ.get("SARVAM_API_KEY", ""):
    print("No key set. Skipping the transliteration call.")
    for place in sample:
        print(f"  {place.native:<28} {place.name}  ({place.language_code})")
else:
    projected = sarvam_projection.project_roster(sample)
    for place in sample:
        print(f"  {place.native:<28} {place.name:<28} {projected[place.name]}")

## What this recipe does not do

- **No phonetic key.** For the query "Kanpur", raw edit similarity ranks Kannur in Kerala (0.833)
  above both Kanpur districts (0.706). The band system contains the damage, since the answer is ASK
  and the question does name both Kanpurs, but the ranking is wrong. A phonetic or token-aware
  score is what would fix it, and it is named here rather than half-built.
- **No telephony, no coordinates, no maps.** One clip in, one district and a state out.
- **No full district list for India,** and no village-level data. 48 entries, chosen to be hard.
- **No text to speech.** Reading the question back to the caller is a different recipe.

The design, the measurements behind every threshold, and the sources for every rename are in
`docs/specs/kisan-village-matcher.md`.